In [1]:
import pandas as pd
import numpy as np

log_returns = pd.read_csv("data/log_returns.csv", index_col=0, parse_dates=True)
features_aligned = pd.read_csv("data/features_aligned.csv", index_col=0, parse_dates=True)
target = pd.read_csv("data/target.csv", index_col=0, parse_dates=True)

print(log_returns.shape)
print(features_aligned.shape)
print(target.shape)

(2764, 462)
(2739, 2772)
(2739, 462)


In [2]:
tickers = log_returns.columns.tolist()
feature_names = ["roll_mean_5", "roll_mean_20", "roll_std_5", "roll_std_20", "mom_5", "mom_20"]
n_timesteps = len(features_aligned)
n_stocks = len(tickers)
n_features = len(feature_names)

# reshape wide dataframe into (timesteps, stocks, features)
raw = np.zeros((n_timesteps, n_stocks, n_features))

for i, ticker in enumerate(tickers):
    for j, feat in enumerate(feature_names):
        col = f"{ticker}_{feat}"
        raw[:, i, j] = features_aligned[col].values

print("Raw feature tensor shape:", raw.shape)

# cross-sectional normalization: at each timestep, normalize across stocks
features_norm = np.zeros_like(raw)

for t in range(n_timesteps):
    for j in range(n_features):
        col = raw[t, :, j]
        mean = col.mean()
        std = col.std()
        if std > 0:
            features_norm[t, :, j] = (col - mean) / std
        else:
            features_norm[t, :, j] = 0.0

print("Normalized feature tensor shape:", features_norm.shape)

Raw feature tensor shape: (2739, 462, 6)
Normalized feature tensor shape: (2739, 462, 6)


In [4]:
lookback = 60
returns_array = log_returns.values  # shape (total_days, 462)

# we only need graphs for timesteps that exist in features_aligned
# features_aligned index is the timestep index
# log_returns may have more rows (features drop first ~20 rows due to rolling)
# align them
aligned_dates = features_aligned.index
returns_aligned = log_returns.loc[aligned_dates].values  # (n_timesteps, n_stocks)

threshold = 0.5
edge_indices = []   # list of (2, num_edges) arrays
edge_weights = []   # list of (num_edges,) arrays

for t in range(n_timesteps):
    date = aligned_dates[t]
    pos = log_returns.index.get_loc(date)

    start = max(0, pos - lookback)
    window = log_returns.iloc[start:pos].values

    if window.shape[0] < 2:
        edge_indices.append(np.zeros((2, 0), dtype=np.int64))
        edge_weights.append(np.zeros(0, dtype=np.float32))
        continue

    corr = np.corrcoef(window.T)
    corr = np.nan_to_num(corr, nan=0.0)  # fix zero-variance stocks

    rows, cols = np.where((corr > threshold) & (np.eye(n_stocks) == 0))

    edge_idx = np.stack([rows, cols], axis=0).astype(np.int64)
    edge_wgt = corr[rows, cols].astype(np.float32)

    edge_indices.append(edge_idx)
    edge_weights.append(edge_wgt)

    if t % 250 == 0:
        print(f"t={t}/{n_timesteps}, edges: {edge_idx.shape[1]}")

print("Done building graphs")

t=0/2739, edges: 88440
t=250/2739, edges: 70580


C:\Users\Rylan Wade\Desktop\FinancialAnalytics\Final_Project\.venv\lib\site-packages\numpy\lib\function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\Rylan Wade\Desktop\FinancialAnalytics\Final_Project\.venv\lib\site-packages\numpy\lib\function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


t=500/2739, edges: 12098
t=750/2739, edges: 4188
t=1000/2739, edges: 81080
t=1250/2739, edges: 7490
t=1500/2739, edges: 36804
t=1750/2739, edges: 24222
t=2000/2739, edges: 82918
t=2250/2739, edges: 28742
t=2500/2739, edges: 13154
Done building graphs


In [5]:
dates = features_aligned.index

train_end = pd.Timestamp("2021-01-01")
val_end   = pd.Timestamp("2022-01-01")

train_mask = dates < train_end
val_mask   = (dates >= train_end) & (dates < val_end)
test_mask  = dates >= val_end

train_idx = np.where(train_mask)[0]
val_idx   = np.where(val_mask)[0]
test_idx  = np.where(test_mask)[0]

print(f"Train: {len(train_idx)} timesteps ({dates[train_idx[0]].date()} to {dates[train_idx[-1]].date()})")
print(f"Val:   {len(val_idx)} timesteps ({dates[val_idx[0]].date()} to {dates[val_idx[-1]].date()})")
print(f"Test:  {len(test_idx)} timesteps ({dates[test_idx[0]].date()} to {dates[test_idx[-1]].date()})")

Train: 1490 timesteps (2015-02-03 to 2020-12-31)
Val:   252 timesteps (2021-01-04 to 2021-12-31)
Test:  997 timesteps (2022-01-03 to 2025-12-22)


In [7]:
import os
os.makedirs("data/processed", exist_ok=True)

np.save("data/processed/features_norm.npy", features_norm)
np.save("data/processed/targets.npy", target.values.astype(np.float32))
np.save("data/processed/train_idx.npy", train_idx)
np.save("data/processed/val_idx.npy", val_idx)
np.save("data/processed/test_idx.npy", test_idx)

# save edge indices and weights as object arrays (variable length per timestep)
edge_indices_arr = np.empty(len(edge_indices), dtype=object)
for i, e in enumerate(edge_indices):
    edge_indices_arr[i] = e

edge_weights_arr = np.empty(len(edge_weights), dtype=object)
for i, w in enumerate(edge_weights):
    edge_weights_arr[i] = w

np.save("data/processed/edge_indices.npy", edge_indices_arr)
np.save("data/processed/edge_weights.npy", edge_weights_arr)

# save tickers list for reference
pd.Series(tickers).to_csv("data/processed/tickers.csv", index=False)

print("Saved all processed data")

Saved all processed data


In [8]:
print("features_norm:", features_norm.shape)     # (2739, 462, 6)
print("targets:", target.values.shape)            # (2739, 462)
print("train_idx:", train_idx.shape)
print("val_idx:", val_idx.shape)
print("test_idx:", test_idx.shape)

# check no NaNs
print("NaNs in features:", np.isnan(features_norm).sum())
print("NaNs in targets:", np.isnan(target.values).sum())

# check a sample graph
t = train_idx[100]
print(f"\nSample graph at t={t}:")
print(f"  edges: {edge_indices[t].shape}")
print(f"  avg edge weight: {edge_weights[t].mean():.3f}")

# check target distribution
flat_targets = target.values[train_idx].flatten()
print(f"\nTarget stats (train):")
print(f"  mean: {flat_targets.mean():.4f}")
print(f"  std:  {flat_targets.std():.4f}")
print(f"  min:  {flat_targets.min():.4f}")
print(f"  max:  {flat_targets.max():.4f}")

features_norm: (2739, 462, 6)
targets: (2739, 462)
train_idx: (1490,)
val_idx: (252,)
test_idx: (997,)
NaNs in features: 0
NaNs in targets: 0

Sample graph at t=100:
  edges: (2, 16778)
  avg edge weight: 0.582

Target stats (train):
  mean: 0.0024
  std:  0.0452
  min:  -1.1431
  max:  0.7806
